# Day 1 模块 5：基础统计与分组分析

这一模块的重点：从“读表”进入“分析问题”。

前面我们已经把表读进来了，现在要开始像小店老板一样提问题：

- 这家店整体卖得怎么样？
- 哪些情况下营收更高？
- 我们能不能从数据里找到一些规律？

In [8]:
from pathlib import Path
import pandas as pd

candidate_paths = [Path('day1_cafe_sales.csv'), Path('day1') / 'day1_cafe_sales.csv', Path('教学课程') / 'day1' / 'day1_cafe_sales.csv']
for path in candidate_paths:
    if path.exists():
        csv_path = path
        break
else:
    raise FileNotFoundError('未找到 day1_cafe_sales.csv')
df = pd.read_csv(csv_path)

## 1. 先看整体情况

如果你是咖啡馆老板，通常会先问几个最直观的问题：

- 一共有多少天数据？
- 平均每天营收多少？
- 最高和最低营收分别是多少？

这一步还不是深分析，而是先对整体情况有一个基本感觉。

In [9]:
summary = {
    '总天数': len(df),
    '平均营收': round(df['sales'].mean(), 2),
    '最高营收': int(df['sales'].max()),
    '最低营收': int(df['sales'].min()),
}
for key, value in summary.items():
    print(f'{key}: {value}')

总天数: 200
平均营收: 2751.16
最高营收: 5939
最低营收: 1291


## 2. 先换成业务问题

分析不是为了算数字，而是为了回答问题。

咖啡馆老板可能会问：

> 不同天气下，平均营收一样吗？
> 如果晴天卖得更好，下次晴天要不要多备一点货？

现在我们就试着用数据回答这个问题。

## 2.1 `groupby()` 示意图

先看这张图，再理解代码会轻松很多。

![groupby 分组统计示意图](day1_groupby_explained_demo.png)

先记住一句话：

- `groupby()`：先把同类放一起
- `mean()`：再给每一组算平均数
- 连起来就是“分组后做统计”

In [10]:
# 按天气分组，查看销售额的基本统计量
grouped = df.groupby('weather_label')['sales']
print(grouped.describe())

               count         mean         std     min      25%     50%  \
weather_label                                                            
多云              43.0  2828.883721  712.585875  1705.0  2243.50  2854.0   
大雨              16.0  2345.625000  606.307279  1393.0  1931.75  2106.0   
小雨              23.0  2439.695652  586.461534  1486.0  2041.50  2406.0   
晴               80.0  2940.887500  829.166792  1644.0  2333.00  2787.0   
阴               38.0  2623.052632  840.234811  1291.0  2065.50  2404.0   

                   75%     max  
weather_label                   
多云             3253.00  4210.0  
大雨             2855.50  3341.0  
小雨             2814.50  3771.0  
晴              3413.25  5939.0  
阴              2973.25  4840.0  


In [11]:
weather_order = ['晴', '多云', '阴', '小雨', '大雨']
avg_sales_by_weather = df.groupby('weather_label')['sales'].mean().reindex(weather_order).round(2)
avg_sales_by_weather

weather_label
晴     2940.89
多云    2828.88
阴     2623.05
小雨    2439.70
大雨    2345.62
Name: sales, dtype: float64

In [14]:
# 正确写法：使用字典指定每列的聚合方式
weekend_stats = df.groupby('is_weekend').agg({
    'sales': ['count', 'mean', 'std', 'min', 'median', 'max']
}).round(2)

weekend_stats.index = ['工作日', '周末']
print(weekend_stats)

    sales                                     
    count     mean     std   min  median   max
工作日   144  2689.16  805.78  1291  2499.0  5939
周末     56  2910.59  721.67  1786  2832.5  4840


In [16]:
# 按是否周末分组，查看销售额基本统计量
weekend_stats = df.groupby('is_weekend')['sales'].agg([
    ('样本数', 'count'),
    ('平均值', 'mean'),
    ('标准差', 'std'),
    ('最小值', 'min'),
    ('中位数', 'median'),
    ('最大值', 'max')
]).round(2)

# 重命名索引，让结果更清晰
weekend_stats.index = ['工作日', '周末']

print("=== 周末 vs 工作日销售额统计 ===")
print(weekend_stats)

=== 周末 vs 工作日销售额统计 ===
     样本数      平均值     标准差   最小值     中位数   最大值
工作日  144  2689.16  805.78  1291  2499.0  5939
周末    56  2910.59  721.67  1786  2832.5  4840


## 3. 再问一个问题

老板还可能会关心：

> 周末和工作日的平均营收一样吗？
> 如果周末更高，是不是要多安排一点人手？

现在继续用分组统计来回答。

In [12]:
avg_sales_by_weekend = df.groupby('is_weekend')['sales'].mean().round(2)
avg_sales_by_weekend

is_weekend
0    2689.16
1    2910.59
Name: sales, dtype: float64

## 4. 再问一个问题

另一个很现实的问题是：

> 做促销到底值不值？
> 促销时的平均营收，会不会比不促销更高？

这也是数据分析经常要回答的问题。

In [13]:
avg_sales_by_promotion = df.groupby('promotion')['sales'].mean().round(2)
avg_sales_by_promotion

promotion
0    2571.79
1    3275.20
Name: sales, dtype: float64

## 5. 模块小结

这一模块先记住三个核心想法：

- `mean()` 是平均数
- `groupby()` 是先分组，再统计
- 数据分析本质上是在回答业务问题

也就是说，我们不是为了“会写代码”才去算这些数，
而是为了回答：天气、周末、促销这些因素，会不会影响营收。